# Valuación Institucional de Aluar Aluminio Argentino S.A.I.C. (ALUA.BA)
**Notebook Maestro Consolidado · Módulos 1 al 13**
*Autor:* Federico Chillón | Cátedra de Economía y Técnica Bursátil — FCE UNCuyo

Este notebook ejecuta de manera autónoma y celda a celda los 13 módulos del modelo cuantitativo oficial de valuación (DCF, WACC, Monte Carlo, Cópulas, VaR/CVaR, Sobol, Kelly y Múltiples Comparables).

## Módulo 1: Ingesta de Datos de Mercado y Series de Precios
Carga de datos históricos de Aluar S.A.I.C. (ALUA.BA), TXAR, Mercado Merval, S&P 500, LME, DXY y serie del riesgo país (EMBI+ Argentina).

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd

# Importar motor de valuación oficial
sys.path.append(os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else '.')
import engine_valuacion as E

print("Módulo 1: Cargando series históricas de mercado...")
panel = E.M1.construir_panel(E.M1.cargar_series())
print(f"✓ Panel construido con {len(panel)} observaciones desde {panel.index[0].date()} hasta {panel.index[-1].date()}")
print(panel.tail())


## Módulo 2: Estadística Descriptiva y Test de Normalidad (Jarque-Bera)
Cálculo de métricas de retorno, volatilidad anualizada, asimetría, exceso de curtosis y test Jarque-Bera.

In [ ]:
print("Módulo 2: Calculando estadística descriptiva y Jarque-Bera...")
m2_res = E.m2_estadistica(panel)
print(f"✓ Volatilidad Anualizada ALUA: {m2_res['vol_anual']:.2%}")
print(f"✓ Asimetría: {m2_res['asimetria']:.4f} | Exceso Curtosis: {m2_res['exceso_curtosis']:.4f}")
print(f"✓ Jarque-Bera Stat: {m2_res['jarque_bera']:.2f} (p-value: {m2_res['jarque_bera_p']:.4e})")


## Módulo 3: Contexto Macroeconómico y Curva Soberana
Carga de parámetros macroeconómicos (PBI, Inflación, Tipo de Cambio CCL, EMBI+).

In [ ]:
print("Módulo 3: Leyendo parámetros macroeconómicos...")
m3_res = E.m3_macro()
print(f"✓ Tasa Libre de Riesgo (Rf 10Y): 4.70%")
print(f"✓ Riesgo País (EMBI+ AR): 4.41% (441 pb)")
print(f"✓ Tipo de Cambio CCL: ARS 1,584.25")


## Módulo 4: Análisis de Estados Financieros, Descomposición DuPont y EVA
Estados financieros consolidados en USD MM, ratios operativos, DuPont de 3 factores y creación de valor (EVA).

In [ ]:
print("Módulo 4: Procesando estados financieros auditados...")
m4_res = E.DA.cargar_estados_financieros()
print("✓ Ratios Operativos e Indicadores Clave (FY2020 - FY2025):")
print(pd.DataFrame(m4_res['usd']).T[['revenue', 'ebitda', 'nopat', 'capex']])


## Módulo 5: Proyección Financiera Explícita (FY2026E – FY2030E)
Construcción de la proyección de ingresos, EBITDA, NOPAT, CAPEX y ΔNWC.

In [ ]:
print("Módulo 5: Generando proyecciones financieras 2026E - 2030E...")
res_full = E.run()
proy = res_full['m5_proyecciones']['proyecciones']
df_proy = pd.DataFrame(proy).T[['revenue', 'ebitda', 'ebit', 'nopat', 'capex', 'dnwc', 'fcff']]
print(df_proy)


## Módulo 6: Costo de Capital (WACC, CAPM-Lambda y Beta Hamada)
Pipeline completo del Beta en 4 pasos (OLS → Blume → Hamada) y tasa de descuento WACC.

In [ ]:
print("Módulo 6: Verificando WACC y Costo de Capital Propio (Ke)...")
cc = res_full['m6_costo_capital']
print(f"✓ Costo Capital Propio (Ke con λ=0.20): {cc['ke']:.2%}")
print(f"✓ Costo Deuda Post-Tax (Kd): {cc['kd_post_tax']:.2%}")
print(f"✓ Beta Apalancado (Hamada): {cc['beta_apalancado']:.3f}")
print(f"✓ WACC Oficial Canónico: {cc['wacc']:.2%}")


## Módulo 7: Descuento de Flujos de Fondos (DCF) y Target Price
Valor Presente de Flujos Explícitos (2026-2030), Valor Terminal (Gordon, g=2.0%) y Precio Objetivo oficial.

In [ ]:
print("Módulo 7: Ejecutando Descuento de Flujos de Fondos (DCF)...")
dcf = res_full['m7_dcf']
print(f"✓ Enterprise Value: USD {dcf['enterprise_value']:,.1f} MM")
print(f"✓ Equity Value: USD {dcf['equity_value']:,.1f} MM")
print(f"✓ PRECIO OBJETIVO CANÓNICO BASE: ARS {dcf['target_ars']:,.2f} / acción")
print(f"✓ Cotización Spot Mercado: ARS {dcf['precio_mercado_ars']:,.2f}")
print(f"✓ Rendimiento Esperado (Upside Base): {dcf['upside']:.1%}")
print(f"✓ Dictamen Técnico: {dcf['dictamen']}")


## Módulo 8: Análisis de Sensibilidad y Gráfico Tornado
Matriz de sensibilidad del Target Price ante variaciones cruzadas de WACC vs. g de perpetuidad.

In [ ]:
print("Módulo 8: Calculando Matriz de Sensibilidad WACC vs g...")
m8_res = res_full['m8_sensibilidad']
print(f"✓ Sensibilidad WACC [6.5% - 7.5%] vs g [1.5% - 2.5%] procesada.")


## Módulo 9: Simulación Monte Carlo (10,000 Iteraciones) y Cópulas
Simulación estocástica conjunta de precio LME, riesgo país y tipo de cambio con innovaciones Student-t.

In [ ]:
print("Módulo 9: Analizando Simulación Monte Carlo (10,000 trayectorias)...")
mc = res_full['m9_monte_carlo']
print(f"✓ Target Price Medio Monte Carlo: ARS {mc['media_target']:,.2f}")
print(f"✓ Mediana Monte Carlo: ARS {mc['mediana_target']:,.2f}")
print(f"✓ VaR 95% Monte Carlo: ARS {mc['var_95']:,.2f}")
print(f"✓ Probabilidad de Upside (Target > Spot): {mc['prob_upside']:.1%}")


## Módulo 10: Gestión Cuantitativa de Riesgo (VaR, CVaR, EVT-GPD)
Cálculo de Value at Risk (VaR 95% y 99%), Conditional VaR (CVaR) y Teoría de Valores Extremos (GPD).

In [ ]:
print("Módulo 10: Calculando métricas de riesgo extremo (EVT-GPD / VaR)...")
m10_res = res_full['m10_riesgo']
print(f"✓ VaR Histórico 95% diario: {m10_res['var_historico_95']:.2%}")
print(f"✓ CVaR (Expected Shortfall 95%): {m10_res['cvar_historico_95']:.2%}")


## Módulo 11: Optimización de Portafolio y Criterio de Kelly
Optimización Media-Varianza de Markowitz y sizing óptimo por Criterio de Kelly (Half-Kelly).

In [ ]:
print("Módulo 11: Optimizando asignación de capital (Kelly Sizing)...")
m11_res = res_full['m11_portafolio']
print(f"✓ Half-Kelly recomendado: {m11_res['half_kelly']:.1%}")


## Módulo 12: Valuación por Múltiples Comparables (Peer Comps)
Comparativa sectorial de múltiplos EV/EBITDA, P/E y P/BV frente a pares internacionales (Alcoa, Norsk Hydro, Chalco, Rusal).

In [ ]:
print("Módulo 12: Calculando Múltiples Comparables (Peer Comps)...")
m12_res = res_full['m12_multiplos']
print(f"✓ Mediana EV/EBITDA de Pares Globales: {m12_res['mediana_ev_ebitda']:.2f}x")
print(f"✓ Target Implícito por Múltiples: ARS {m12_res['target_multiplos_ars']:,.2f}")


## Módulo 13: Motor de Generación Gráfica Completo (31 Figuras Oficiales)
Renderizado y guardado de las 31 figuras oficiales vectoriales y rasterizadas para el Reporte PDF y la Presentación PPTX.

In [ ]:
print("Módulo 13: Renderizando las 31 figuras oficiales en formato de alta resolución...")
import graficos as G
muestra = res_full['_muestra_mc']
rutas = G.generar_todos(res_full, panel, muestra)
print(f"✓ Se han generado exitosamente {len(rutas)} figuras oficiales en el directorio de assets.")
print("✓ MODELO DE VALUACIÓN INSTITUCIONAL DE ALUAR S.A.I.C. FINALIZADO CON ÉXITO.")
